# 10 — Multi-Task PPO: Train on 1-1 and 1-2 Simultaneously

Trains a single PPO agent on **both** World 1-1 and World 1-2 at the same time,
avoiding catastrophic forgetting.

**Strategy:**
- Start from the pre-trained 1-1 model (already knows 1-1)
- 8 parallel envs: **2 on 1-1** (retention) + **6 on 1-2** (learning)
- More 1-2 envs so the harder level dominates the gradient
- Higher entropy (0.02) to encourage exploration on the new level
- The 1-1 envs act as "regularization" against catastrophic forgetting

**Comparison:**
- Single-task 1-1 → wins 1-1, fails 1-2
- Transfer 1-1→1-2 → wins 1-2, forgets 1-1 (catastrophic forgetting)
- **Multi-task** → should win both levels

In [1]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

/home/lmartim4/Desktop/CSC-52081-Mario


In [2]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_symbolic_multitask_vec_env, make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback

print(f'Using CUDA: {torch.cuda.is_available()}')

Using CUDA: True


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
# Create 8 parallel envs: 2 on World 1-1 (retention), 6 on World 1-2 (learning)
LEVELS = ['SuperMarioBros-1-1-v3', 'SuperMarioBros-1-2-v3']
ENVS_PER_LEVEL = [2, 6]  # asymmetric: more weight on the harder level

env = make_symbolic_multitask_vec_env(
    env_ids=LEVELS,
    skip=4,
    n_stack=4,
    flatten=True,
    envs_per_level=ENVS_PER_LEVEL,
)

total_envs = sum(ENVS_PER_LEVEL)
print(f'Levels: {LEVELS}')
print(f'Envs per level: {ENVS_PER_LEVEL}')
print(f'Total parallel envs: {total_envs}')
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: como faz para ver o mario{env.action_space.n}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Levels: ['SuperMarioBros-1-1-v3', 'SuperMarioBros-1-2-v3']
Envs per level: [2, 6]
Total parallel envs: 8
Observation space: (832,)
Action space: como faz para ver o mario7


/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [4]:
# Load pre-trained 1-1 model and continue training on both levels
model = PPO.load(
    'models/symbolic_ppo/final_model',
    env=env,
    device='cpu',
    learning_rate=2.5e-4,
    ent_coef=0.02,  # higher entropy to encourage exploration on 1-2
)

TOTAL_STEPS = 4_000_000

print(f'Loaded pre-trained 1-1 model')
print(f'Training: {TOTAL_STEPS:,} steps on both levels')
print(f'Device: {model.device}')
print(f'Entropy coef: {model.ent_coef} (higher for exploration)')
print(f'Batch per rollout: {512 * total_envs} samples')

FileNotFoundError: [Errno 2] No such file or directory: 'models/symbolic_ppo/final_model.zip'

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/symbolic_ppo_multitask

In [ ]:
# Train
callback = CheckpointAndLogCallback(
    save_path='models/symbolic_ppo_multitask',
    save_freq=100_000,
)

model.learn(
    total_timesteps=TOTAL_STEPS,
    callback=callback,
    log_interval=1,
)
model.save('models/symbolic_ppo_multitask/final_model')
print('Multi-task training complete!')

## Evaluate on each level separately

In [ ]:
import numpy as np
from stable_baselines3 import PPO

eval_model = PPO.load('models/symbolic_ppo_multitask/final_model')

for level in LEVELS:
    print(f'\n{"="*50}')
    print(f'Evaluating on: {level}')
    print(f'{"="*50}')

    eval_env = make_symbolic_env(
        env_id=level, skip=4, n_stack=4, flatten=True,
    )

    rewards, flags = [], []
    for ep in range(10):
        obs = eval_env.reset()
        obs = obs[0] if isinstance(obs, tuple) else obs
        done, total_reward, flag = False, 0.0, False

        while not done:
            action, _ = eval_model.predict(obs, deterministic=True)
            result = eval_env.step(int(action))
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                done = terminated or truncated
            else:
                obs, reward, done, info = result
            total_reward += float(reward)
            if isinstance(info, dict) and info.get('flag_get', False):
                flag = True

        rewards.append(total_reward)
        flags.append(flag)
        status = 'FLAG!' if flag else 'DEAD'
        print(f'  Ep {ep+1}: reward={total_reward:.1f} [{status}]')

    print(f'\n  Mean reward: {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}')
    print(f'  Flag rate: {np.mean(flags):.0%}')
    eval_env.close()

In [ ]:
# Watch: python watch_agent.py --model models/symbolic_ppo_multitask/final_model --env SuperMarioBros-1-1-v3
# Watch: python watch_agent.py --model models/symbolic_ppo_multitask/final_model --env SuperMarioBros-1-2-v3